## DATA CLEANSING NOTEBOOK

In [1]:
import os

In [2]:
curr_path = !pwd
curr_path = curr_path[0] + "/"
input_file = os.path.join(os.path.dirname(curr_path), '..', 'inputs', 'inventory_raw.csv')
input_file

'/Users/luisenrique/Documents/network-inventory-cleaning-and-validation/notebooks/../inputs/inventory_raw.csv'

In [3]:
import re
import pandas as pd
import numpy as np
import ipaddress

from urllib.parse import urlparse

In [4]:
df_raw = pd.read_csv(input_file)
df_raw

,source_row_id,ip,hostname,fqdn,mac,owner,device_type,site,notes
0,1,192.168.010.005,HOST01,NaN,AA-BB-CC-DD-EE-FF,priya (platform) priya@corp.example.com,server,BLR Campus,db host
1,2,10.0.1.300,host-02,host-02.local,11-22-33-44-55-66,ops,NaN,HQ Bldg 1,edge gw?
2,3,10.0.1,host03,NaN,aabb.ccdd.eeff,jane@corp.example.com,switch,HQ-BUILDING-1,NaN
3,4,10.0.1.1.2,printer-01,NaN,00:11:22:33:44:55,Facilities,printer,HQ,NaN
4,5,fe80::1%eth0,iot-cam01,NaN,00:aa:bb:cc:dd:ee,sec,iot,Lab-1,camera PoE on port 3
5,6,127.0.0.1,local-test,NaN,NaN,NaN,NaN,NaN,NaN
6,7,169.254.10.20,host-apipa,NaN,NaN,NaN,NaN,NaN,NaN
7,8,10.10.10.10,srv-10,NaN,NaN,platform,server,BLR campus,NaN
8,9,abc.def.ghi.jkl,badhost,NaN,NaN,NaN,NaN,NaN,NaN
9,10,192.168.1.-1,neg,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
df_raw.columns

Index(['source_row_id', 'ip', 'hostname', 'fqdn', 'mac', 'owner',
       'device_type', 'site', 'notes'],
      dtype='object')

### IP

IP VERSION

In [ ]:
# TODO: IMPLEMENT IP_VERSION CLASSIFICATION
# def detect_ip_version(ip_str):
#     if pd.isna(ip_str):
#         return "invalid"
#     s = str(ip_str).strip()

#     if s == "":
#         return "invalid"
    
#     colon_count = s.count(':')
#     dot_count = s.count('.')

#     # Ipv6 indicators
#     if colon_count >= 2:
#         try:
#             ipaddress.IPv6Address(s)
#             return "6"
#         except ipaddress.AddressValueError as e:
#             return "invalid"
        
#     elif dot_count == 3:
#         try:
#             ipaddress.IPv4Address(s)
#             return "4"
#         except ipaddress.AddressValueError as e:
#             return "invalid"
        
#     else:
#         return "invalid"

VALIDATION & NORMALIZATION

In [ ]:
def traceability_metadata(field: str, from_value, to_value, reason: str):
    return {
        "field": field,
        "from": from_value,
        "to": to_value,
        "reason": reason
    }

In [ ]:
def ipv4_validate_and_normalize(ip_str):
    # FIELD="ip"
    # FROM=ip_str

    if pd.isna(ip_str):
        return (False, None, "missing")
    s = str(ip_str).strip()
    if ':' in s:
        return (False, None, "ipv6_or_non_ipv4")
    parts = s.split(".")
    if len(parts) != 4:
        return (False, None, "wrong_octet_count")
    
    canonical_parts = []
    for p in parts:
        if p == '':
            return (False, None, "empty_octet")
        if not (p.lstrip("+").isdigit() and not p.startswith("-")):
            return (False, None, "non_numeric_or_negative")
        try:
            v = int(p, 10)
        except ValueError:
            return (False, None, "non_decimal_format")
        if v < 0 or v > 255:
            return (False, None, "octet_out_of_range")
        canonical_parts.append(str(v))
    canonical = ".".join(canonical_parts)

    # TODO: TRACEABILITY STEP
    
    return (True, canonical, "ok")

IP_TYPE

In [8]:
def classify_ipv4_type(ip):
    if pd.isna(ip) or ip is None:
        return "invalid"
    
    try:
        octets = list(map(int, ip.split('.')))

        # Checking private ranges
        if octets[0] == 10:
            return "private_rfc1918"
        elif octets[0] == 172 and 16 <= octets[1] <= 31:
            return "private_rfc1918"
        elif octets[0] == 192 and octets[1] == 168:
            return "private_rfc1918"
        
        # Other ranges
        elif octets[0] == 169 and octets[1] == 254:
            return "link_local_apipa"
        elif octets[0] == 127:
            return "loopback"
        
        return "public_or_other"
    except (ValueError, AttributeError, IndexError):
        return "invalid_format"

SUBNET_CIDR

In [9]:
def default_subnet(ip, ip_type=None):
    if pd.isna(ip) or ip is None:
        return ""
    
    try:
        octets = list(map(int, ip.split('.')))

        # Get Ip type, if not provided
        if ip_type is None:
            ip_type = classify_ipv4_type_pandas(ip)

        # Subnet strategies
        if ip_type == 'private_rfc1918':
            if octets[0] == 10:
                return f"10.0.0.0/8"
            elif octets[0] == 172:
                return f"172.{octets[1]}.0.0/16"
            elif octets[0] == 192 and octets[1] == 168:
                return f"192.168.{octets[2]}.0/24"
            
        elif ip_type == "link_local_apipa":
            return "169.254.0.0/16"

        elif ip_type == "loopback":
            return "127.0.0.0/8"
        
        elif ip_type == "public_or_other":
            # Assumign /24 for public IPs
            return f"{octets[0]}.{octets[1]}.{octets[2]}.0/24"
        
        return ""
    
    except (ValueError, AttributeError, IndexError):
        return ""

In [10]:
df_final = pd.DataFrame()
df_ipv4 = pd.DataFrame()
df_ipv4[['ip_valid', 'ip_canonical', 'ip_reason']] = df_raw['ip'].apply(
    lambda x: pd.Series(ipv4_validate_and_normalize(x))
)
df_ipv4['ip_type'] = df_ipv4['ip_canonical'].apply(
    lambda x: pd.Series(classify_ipv4_type(x))
)
df_ipv4['subnet_cidr'] = df_ipv4.apply(
    lambda x: default_subnet(x['ip_canonical'], x['ip_type'])
    if x['ip_valid'] else "",
    axis=1
)
df_final[['ip', 'ip_valid', 'ip_type', 'subnet_cidr']] = df_ipv4[['ip_canonical', 'ip_valid', 'ip_type', 'subnet_cidr']]
df_final['hostname'] = df_raw['hostname']
df_final

,ip,ip_valid,ip_type,subnet_cidr,hostname
0,192.168.10.5,True,private_rfc1918,192.168.10.0/24,HOST01
1,None,False,invalid,,host-02
2,None,False,invalid,,host03
3,None,False,invalid,,printer-01
4,None,False,invalid,,iot-cam01
5,127.0.0.1,True,loopback,127.0.0.0/8,local-test
6,169.254.10.20,True,link_local_apipa,169.254.0.0/16,host-apipa
7,10.10.10.10,True,private_rfc1918,10.0.0.0/8,srv-10
8,None,False,invalid,,badhost
9,None,False,invalid,,neg


### HOSTNAME

VALIDATION & NORMALIZATION

In [ ]:
def validate_hostname(hostname_str):
    """Validating hostname according to RFC1123"""
    if pd.isna(hostname_str):
        return (False, None, "missing")
    
    s = str(hostname_str).strip()

    if s == "":
        return (False, None, "empty_string")
    
    if len(s) > 63:
        return (False, s, "too_long")
    
    # Checking periods
    if '.' in s:
        return (False, s, "contains_periods")
    
    # Character validation - RFC 1123
    if not re.match(r'^[a-zA-Z0-9]([a-zA-Z0-9-]*[a-zA-Z0-9])?$', s):
        return (False, s, "invalid_characters")
    
    # Cannot start or end with hyphen
    if s.startswith('-') or s.endswith('-'):
        return (False, s, "hyphen_at_edge")
    
    # Normalize to lowercase for consistency
    canonical = s.lower()

    # TODO: TRACEABILITY STEP
    return (True, canonical, "ok")
    

In [12]:
df_hostname = pd.DataFrame()
df_hostname['hostname'] = df_raw['hostname']
df_hostname[['hostname_valid', 'hostname_canonical', 'hostname_reason']] = df_hostname['hostname'].apply(
    lambda x: pd.Series(validate_hostname(x))
)
df_hostname

,hostname,hostname_valid,hostname_canonical,hostname_reason
0,HOST01,True,host01,ok
1,host-02,True,host-02,ok
2,host03,True,host03,ok
3,printer-01,True,printer-01,ok
4,iot-cam01,True,iot-cam01,ok
5,local-test,True,local-test,ok
6,host-apipa,True,host-apipa,ok
7,srv-10,True,srv-10,ok
8,badhost,True,badhost,ok
9,neg,True,neg,ok


### SITE

NORMALIZATION

In [ ]:
def normalize_site(site: str) -> str:
    """
    Normalize a site name string and return a standardized form.
    
    Examples:
      'BLR Campus'    -> 'blr-campus'
      'HQ Bldg 1'     -> 'hq-bldg-1'
      'HQ-BUILDING-1' -> 'hq-bldg-1'
      'Lab-1'         -> 'lab-1'
      NaN or None     -> 'unknown'
    """
    
    if pd.isna(site) or site is None:
        return "unknown"
    
    s = str(site).strip().lower()
    if s == "":
        return "unknown"
    
    # Replace underscores, multiple spaces, and commas with hyphens
    s = re.sub(r"[\s,_]+", "-", s)
    
    # Common standardization: building → bldg, campus → campus (unchanged)
    s = s.replace("building", "bldg")
    
    # Remove duplicate hyphens
    s = re.sub(r"-{2,}", "-", s)
    
    # Strip trailing or leading hyphens
    s = s.strip("-")
    
    # Handle very generic cases
    if s in {"n/a", "na", "none", "null", "unknown"}:
        return "unknown"
    
    # TODO: TRACEABILITY STEP
    
    return s

In [14]:
df_site = pd.DataFrame()

df_site['original_site'] = df_raw['site']
df_site['site_normalized'] = df_raw['site'].apply(
    lambda x: pd.Series(normalize_site(x))
)
df_site

,original_site,site_normalized
0,BLR Campus,blr-campus
1,HQ Bldg 1,hq-bldg-1
2,HQ-BUILDING-1,hq-bldg-1
3,HQ,hq
4,Lab-1,lab-1
5,NaN,unknown
6,NaN,unknown
7,BLR campus,blr-campus
8,NaN,unknown
9,NaN,unknown


### FQDN

VALDIATION & NORMALIZATION

In [ ]:
def validate_fqdn(fqdn_str, hostname_part=None, site_part=None):
    """
    Validates FQDN according to RFC 1035 and checks consistency with hostname/site.
    Returns a tuple: (is_valid: bool, normalized_value: str, error_code: str, consistency: str)
    """
    if pd.isna(fqdn_str) or fqdn_str is None:
        return (False, None, "missing", "inconsistent")
    
    s = str(fqdn_str).strip()

    if s == "":
        return (False, None, "empty_string", "inconsistent")
    
    # Remove trailing dot if present (optional in some systems)
    if s.endswith('.'):
        s = s[:-1]
    
    # Check overall length (RFC 1035: max 255 chars including dots)
    if len(s) > 255:
        return (False, s, "too_long", "inconsistent")
    
    # Split into labels
    labels = s.split('.')
    
    # Must have at least 2 labels (hostname + domain)
    if len(labels) < 2:
        return (False, s, "too_few_labels", "inconsistent")
    
    # Validate each label
    for i, label in enumerate(labels):
        # Check label length (max 63 chars per RFC 1035)
        if len(label) > 63:
            return (False, s, f"label_{i}_too_long", "inconsistent")
        
        # Check for empty labels (consecutive periods)
        if len(label) == 0:
            return (False, s, "empty_label", "inconsistent")
        
        # First and last label have special rules
        if i == 0:
            # First label is essentially the hostname - apply hostname rules
            if not re.match(r'^[a-zA-Z0-9]([a-zA-Z0-9-]*[a-zA-Z0-9])?$', label):
                return (False, s, "invalid_hostname_label", "inconsistent")
        else:
            # Other labels (domain parts) can start/end with digits
            # But still no leading/trailing hyphens
            if not re.match(r'^[a-zA-Z0-9]([a-zA-Z0-9-]*[a-zA-Z0-9])?$', label):
                # Allow digits at start/end for domain labels (like "2ndfloor" or "lab1")
                if not re.match(r'^[a-zA-Z0-9-]+$', label):
                    return (False, s, f"invalid_domain_label_{i}", "inconsistent")
        
        # Check for leading/trailing hyphens in all labels
        if label.startswith('-') or label.endswith('-'):
            return (False, s, f"label_{i}_hyphen_edge", "inconsistent")
    
    # Normalize to lowercase
    canonical = s.lower()
    
    # Check consistency with hostname and site if provided
    consistency_status = "unknown"
    if hostname_part is not None and site_part is not None:
        expected_fqdn = f"{hostname_part}.{site_part}".lower()
        if canonical == expected_fqdn:
            consistency_status = "consistent"
        else:
            consistency_status = "inconsistent"

    # TODO: TRACEABILITY STEP
    
    return (True, canonical, "valid", consistency_status)

In [16]:

def generate_reverse_ptr(ip):
    if pd.isna(ip) or ip is None:
        return None

    s_ip = str(ip).strip()
    if s_ip == "":
        return None

    try:
        ip_obj = ipaddress.ip_address(s_ip)
    except ValueError:
        return None

    reverse_key = ip_obj.reverse_pointer.lower().rstrip('.') 

    return reverse_key

In [17]:
df_fqdn = pd.DataFrame()
df_fqdn['fqdn'] = df_raw['fqdn']
df_fqdn['hostname_canonical'] = df_hostname['hostname_canonical']
df_fqdn['site'] = df_site['site_normalized']
df_fqdn[['reverse_ptr']] = df_final['ip'].apply(
    lambda x: pd.Series(generate_reverse_ptr(x))
)
df_fqdn[['fqdn_valid', 'fqdn_canonical', 'fqdn_error_reason', 'fqdn_consistency']] = (
    df_fqdn.apply(
        lambda x: validate_fqdn(x['fqdn'], x['hostname_canonical'], x['site']),
        axis='columns'
    ).apply(pd.Series)
)
df_fqdn


,fqdn,hostname_canonical,site,reverse_ptr,fqdn_valid,fqdn_canonical,fqdn_error_reason,fqdn_consistency
0,NaN,host01,blr-campus,5.10.168.192.in-addr.arpa,False,None,missing,inconsistent
1,host-02.local,host-02,hq-bldg-1,NaN,True,host-02.local,valid,inconsistent
2,NaN,host03,hq-bldg-1,NaN,False,None,missing,inconsistent
3,NaN,printer-01,hq,NaN,False,None,missing,inconsistent
4,NaN,iot-cam01,lab-1,NaN,False,None,missing,inconsistent
5,NaN,local-test,unknown,1.0.0.127.in-addr.arpa,False,None,missing,inconsistent
6,NaN,host-apipa,unknown,20.10.254.169.in-addr.arpa,False,None,missing,inconsistent
7,NaN,srv-10,blr-campus,10.10.10.10.in-addr.arpa,False,None,missing,inconsistent
8,NaN,badhost,unknown,NaN,False,None,missing,inconsistent
9,NaN,neg,unknown,NaN,False,None,missing,inconsistent


### MAC

NORMALIZATION AND VALIDATION

In [ ]:
def validate_mac(mac_str):
    """
    Validates a MAC address string.
    Returns a tuple: (is_valid: bool, normalized_value: str | None, error_code: str)
    
    Rules:
      - Must not be missing or empty
      - Must match one of the common formats (colon, hyphen, or dot separated)
      - Normalized output: colon-separated lowercase (e.g., 'aa:bb:cc:dd:ee:ff')
    """
    if pd.isna(mac_str) or mac_str is None:
        return (False, None, "missing")
    
    s = str(mac_str).strip()
    if s == "":
        return (False, None, "empty_string")

    # Define regex patterns for common formats
    patterns = [
        r"^([0-9A-Fa-f]{2}[:-]){5}([0-9A-Fa-f]{2})$",   # AA:BB:CC:DD:EE:FF or AA-BB-CC-DD-EE-FF
        r"^([0-9A-Fa-f]{4}\.){2}([0-9A-Fa-f]{4})$",     # AAAA.BBBB.CCCC
    ]
    
    # Check match
    if not any(re.match(p, s) for p in patterns):
        return (False, None, "invalid_format")
    
    # Normalize to colon-separated lowercase format
    s = s.lower().replace('-', ':')
    if '.' in s:
        # Cisco-style -> flatten and regroup
        s = s.replace('.', '')
        s = ':'.join([s[i:i+2] for i in range(0, 12, 2)])

    # TODO: TRACEABILITY STEP
    
    return (True, s, "valid")

In [19]:
df_mac = pd.DataFrame()
df_mac['mac'] = df_raw['mac']
df_mac[['mac_valid', 'mac_canonical', 'mac_reason']] = df_mac['mac'].apply(
    lambda x: pd.Series(validate_mac(x))
)
df_mac

,mac,mac_valid,mac_canonical,mac_reason
0,AA-BB-CC-DD-EE-FF,True,aa:bb:cc:dd:ee:ff,valid
1,11-22-33-44-55-66,True,11:22:33:44:55:66,valid
2,aabb.ccdd.eeff,True,aa:bb:cc:dd:ee:ff,valid
3,00:11:22:33:44:55,True,00:11:22:33:44:55,valid
4,00:aa:bb:cc:dd:ee,True,00:aa:bb:cc:dd:ee,valid
5,NaN,False,None,missing
6,NaN,False,None,missing
7,NaN,False,None,missing
8,NaN,False,None,missing
9,NaN,False,None,missing


### OWNER

PARSING

In [ ]:
def parse_owner(owner_str):
    """
    Parses a single owner string into (owner, owner_email, owner_team).
    
    Returns:
        tuple: (owner, owner_email, owner_team)
    """
    if pd.isna(owner_str) or not str(owner_str).strip():
        return (pd.NA, pd.NA, pd.NA)
    
    s = str(owner_str).strip()
    
    # Regex for email detection
    email_pattern = re.compile(r"[\w\.-]+@[\w\.-]+\.\w+")
    
    # Extract email if present
    email_match = email_pattern.search(s)
    email = email_match.group(0) if email_match else pd.NA
    s_no_email = email_pattern.sub("", s).strip()
    
    # Extract team (inside parentheses)
    team_match = re.search(r"\((.*?)\)", s_no_email)
    team = team_match.group(1).strip() if team_match else pd.NA
    s_no_paren = re.sub(r"\(.*?\)", "", s_no_email).strip()
    
    # Remaining part is likely the owner name or alias
    owner = s_no_paren if s_no_paren else pd.NA

    # TODO: TRACEABILITY STEP FOR OWNER ONLY
    
    return (owner, email, team)

In [21]:
# TODO: MIGHT HAVE TO PASS THIS ONE THROUGH THE LLM
df_owner = pd.DataFrame()
df_owner['original_owner'] = df_raw['owner']
df_owner[['owner', 'owner_email', 'owner_team']] = df_raw['owner'].apply(
    lambda x: parse_owner(x)
).apply(pd.Series)
df_owner

,original_owner,owner,owner_email,owner_team
0,priya (platform) priya@corp.example.com,priya,priya@corp.example.com,platform
1,ops,ops,<NA>,<NA>
2,jane@corp.example.com,<NA>,jane@corp.example.com,<NA>
3,Facilities,Facilities,<NA>,<NA>
4,sec,sec,<NA>,<NA>
5,NaN,<NA>,<NA>,<NA>
6,NaN,<NA>,<NA>,<NA>
7,platform,platform,<NA>,<NA>
8,NaN,<NA>,<NA>,<NA>
9,NaN,<NA>,<NA>,<NA>


### DEVICE_TYPE

CLASSIFICATION

In [ ]:
def normalize_device_type(device_type: str):
    """
    Normalize a device_type string and return a tuple:
      (normalized_device_type: str, device_type_confidence: int)
      
    Rules:
      - 100 → exact match to known canonical type
      - 90  → fuzzy or alias match (e.g. abbreviation, keyword)
      - 0   → unrecognized or missing
    """

    # Canonical device types
    canonical_types = {
        "switch", "router", "firewall", "server", "printer",
        "wireless_ap", "wireless_controller", "load_balancer",
        "storage", "ups", "ip_phone", "camera", "unknown"
    }

    # Strong / exact aliases → canonical
    strong_map = {
        "switch": "switch",
        "router": "router",
        "firewall": "firewall",
        "server": "server",
        "printer": "printer",
        "access_point": "wireless_ap",
        "wireless_ap": "wireless_ap",
        "controller": "wireless_controller",
        "load_balancer": "load_balancer",
        "storage": "storage",
        "ups": "ups",
        "phone": "ip_phone",
        "camera": "camera"
    }

    # Fuzzy patterns for common abbreviations or variations
    fuzzy_patterns = [
        (r"\bsw\b|switch", "switch"),
        (r"\brtr\b|router", "router"),
        (r"\bfw\b|firewall|asa|pa\d+", "firewall"),
        (r"\bap\b|access[_-]?point", "wireless_ap"),
        (r"wlc|controller", "wireless_controller"),
        (r"f5|ltm|lb|load[_-]?balancer", "load_balancer"),
        (r"server|vm|esxi", "server"),
        (r"printer|print", "printer"),
        (r"ups", "ups"),
        (r"voip|phone", "ip_phone"),
        (r"camera|cam", "camera"),
        (r"nas|san|storage", "storage"),
    ]

    # --- Clean input ---
    if pd.isna(device_type) or device_type is None:
        return ("unknown", 0)

    s = str(device_type).strip().lower()
    s = re.sub(r"[,_;/]", " ", s)
    s = re.sub(r"\s+", " ", s)
    if s == "":
        return ("unknown", 0)

    # --- Exact match ---
    if s in strong_map:
        return (strong_map[s], 100)
    if s in canonical_types:
        return (s, 100)

    # --- Fuzzy match ---
    for pattern, normalized in fuzzy_patterns:
        if re.search(pattern, s):
            return (normalized, 90)
        
    # TODO: TRACEABILITY STEP

    # --- No match ---
    return ("unknown", 0)

In [23]:
df_dt = pd.DataFrame()
df_dt['original_dt'] = df_raw['device_type']
df_dt[['device_type', 'device_type_confidence']] = df_raw['device_type'].apply(
    lambda x: normalize_device_type(x)
).apply(pd.Series)

In [24]:
df_dt

,original_dt,device_type,device_type_confidence
0,server,server,100
1,NaN,unknown,0
2,switch,switch,100
3,printer,printer,100
4,iot,unknown,0
5,NaN,unknown,0
6,NaN,unknown,0
7,server,server,100
8,NaN,unknown,0
9,NaN,unknown,0
